# 04 – The business cycle clock (Konjunkturklockan)

**Purpose:** Turn SCB's business cycle clock from monthly data 2000–2026 into a table with **one row per year 2000–2025** and one column per business cycle indicator. The table can then be joined with the fertility data, which is annual.

| | |
|---|---|
| **Source** | SCB, Konjunkturklockan: 14 indicators, two measures, monthly |
| **Raw data** | `data/raw/konjunkturklockan.csv` |
| **Output** | `data/processed/konjunktur_ar.csv` |
| **Dependencies** | None. Can be run at any time. |

The file `CSV-files/konjunkturklockan.csv` in the project has already been filtered by hand. This notebook starts from the **original extract** and does all the filtering in code. That way the steps can be followed and repeated.

### Decisions implemented
1. **Keep only the measure *Konjunkturläge* (business cycle position).** *Förändring från föregående period* (change from previous period) is dropped.
2. **Drop the indicator *BNP kvartal* (quarterly GDP).** It is a quarterly series repeated monthly and is redundant with *BNP-indikator månad* (monthly GDP indicator).
3. **Drop 2026**, which has only 8 months and lies outside the period of the fertility data.
4. **Aggregate months to years** as the mean of the year's 12 months. Incomplete years become `NaN`.

## 1. Settings

Paths as in the other notebooks, plus the period to include. The period follows the fertility data, 2000–2025.

In [1]:
from pathlib import Path

import pandas as pd

BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Cannot find data/raw. Open the notebook from the Assignment folder."

FORSTA_AR = 2000
SISTA_AR = 2025

## 2. Load raw data

Each row is one indicator, one measure and one month. The month is given as text in the format `2000M01`.

In [2]:
konj_ra = pd.read_csv(RAW / "konjunkturklockan.csv", sep=";", encoding="utf-8-sig")

display(konj_ra.head())
print("Shape:", konj_ra.shape)
print(konj_ra.dtypes)

,indikator,tabellinnehåll,månad,Värde
0,BNP-indikator månad,Konjunkturläge,2000M01,-1.07
1,BNP-indikator månad,Konjunkturläge,2000M02,-0.84
2,BNP-indikator månad,Konjunkturläge,2000M03,-0.59
3,BNP-indikator månad,Konjunkturläge,2000M04,-0.29
4,BNP-indikator månad,Konjunkturläge,2000M05,0.01


Shape: (8960, 4)
indikator             str
tabellinnehåll        str
månad                 str
Värde             float64
dtype: object


## 3. Check the structure

We expect **14 indicators × 2 measures × 320 months** (January 2000–August 2026) **= 8,960 rows**, no duplicates and a consistent month format.

The raw data has 134 missing values. Where they are and how they are handled is shown in step 7.

In [3]:
assert list(konj_ra.columns) == ["indikator", "tabellinnehåll", "månad", "Värde"]
assert konj_ra["indikator"].nunique() == 14
assert set(konj_ra["tabellinnehåll"]) == {"Konjunkturläge", "Förändring från föregående period"}
assert konj_ra["månad"].str.fullmatch(r"\d{4}M\d{2}").all()
assert not konj_ra.duplicated(["indikator", "tabellinnehåll", "månad"]).any()
assert len(konj_ra) == 14 * 2 * konj_ra["månad"].nunique()

print("Months:", konj_ra["månad"].min(), "–", konj_ra["månad"].max(), f"({konj_ra['månad'].nunique()} in total)")
print("Missing values:", konj_ra["Värde"].isna().sum())
print("\nIndicators:")
for namn in sorted(konj_ra["indikator"].unique()):
    print("  -", namn)

Months: 2000M01 – 2026M08 (320 in total)
Missing values: 134

Indicators:
  - Arbetade timmar
  - BNP kvartal
  - BNP-indikator månad
  - Detaljhandel sällanköpsvaror
  - Hushållens konfidensindikator
  - Hushållens konsumtion
  - Industrins orderingång
  - Nyregistrerade lastbilar
  - Nyregistrerade personbilar
  - Näringslivets efterfrågan
  - Näringslivets produktion
  - Sysselsättning
  - Varuexport
  - Varuimport


## 4. Keep only the measure Konjunkturläge

The business cycle clock reports two measures for each indicator:

- **Konjunkturläge** (business cycle position): how far the indicator is from its normal level, on a standardised scale where 0 is the normal level. The measure is therefore **already detrended** and needs no further detrending.
- **Förändring från föregående period** (change from previous period): how the position has changed since the previous month.

The project uses the business cycle position, so the change measure is dropped. The check below shows that the values are symmetric around 0, with a median close to 0 and a range of about ±3. This is consistent with a standardised scale.

In [4]:
konj = konj_ra.loc[konj_ra["tabellinnehåll"] == "Konjunkturläge"].drop(columns="tabellinnehåll")

print("Rows remaining:", len(konj))
print(f"Median: {konj['Värde'].median():.2f}   min: {konj['Värde'].min():.2f}   max: {konj['Värde'].max():.2f}")

Rows remaining: 4480
Median: -0.01   min: -3.49   max: 2.92


## 5. Drop BNP kvartal – redundant

*BNP kvartal* (quarterly GDP) is based on the quarterly national accounts but is reported per month. The check below shows two things:

1. Within each quarter, **all three months are exactly equal**. The series has only one new value per quarter and is repeated, not measured monthly.
2. It tracks **BNP-indikator månad** (the monthly GDP indicator) very closely (high correlation). The GDP indicator measures the same thing but has a new value every month.

*BNP kvartal* therefore adds no information and would only produce duplicated rules in ARM. It is dropped.

In [5]:
bnp_kv = konj.loc[konj["indikator"] == "BNP kvartal"].sort_values("månad").reset_index(drop=True)
bnp_kv["kvartal"] = bnp_kv["månad"].str[:4] + "K" + ((bnp_kv["månad"].str[5:].astype(int) - 1) // 3 + 1).astype(str)
olika_per_kvartal = bnp_kv.groupby("kvartal")["Värde"].nunique()

bred = konj.pivot(index="månad", columns="indikator", values="Värde")
korrelation = bred["BNP kvartal"].corr(bred["BNP-indikator månad"])

print("Maximum number of distinct values within a quarter:", olika_per_kvartal.max())
print(f"Correlation with BNP-indikator månad: {korrelation:.3f}")
assert olika_per_kvartal.max() == 1

konj = konj.loc[konj["indikator"] != "BNP kvartal"]
print("Indicators remaining:", konj["indikator"].nunique())

Maximum number of distinct values within a quarter: 1
Correlation with BNP-indikator månad: 0.955
Indicators remaining: 13


## 6. Split the month and drop 2026

The text `2000M01` is split into two integers, `ar` (year) and `manad` (month), so that we can calculate with years.

Then **2026** is dropped for two reasons:

- The year has only 8 months (January–August), and for July–August values are also missing for several indicators. An annual mean of half a year cannot be compared with full years.
- The fertility data ends in 2025, so 2026 cannot be linked to any outcome.

In [6]:
konj = konj.assign(
    ar=konj["månad"].str[:4].astype(int),
    manad=konj["månad"].str[5:].astype(int),
)
assert konj["manad"].between(1, 12).all()

manader_2026 = sorted(konj.loc[konj["ar"] == 2026, "manad"].unique().tolist())
print("Months available for 2026:", manader_2026)

konj = konj.loc[konj["ar"].between(FORSTA_AR, SISTA_AR)]
print("Years remaining:", konj["ar"].min(), "–", konj["ar"].max())

Months available for 2026: [1, 2, 3, 4, 5, 6, 7, 8]
Years remaining: 2000 – 2025


## 7. Missing monthly values

The table shows which months are missing values after filtering. All gaps are at the **start of the series**. Three indicators simply start later than 2000:

- *Arbetade timmar* (hours worked) and *Sysselsättning* (employment) are missing all of 2000.
- *Näringslivets efterfrågan* (business sector demand) is missing all of 2000 and January–November 2001.

So this is not an error in the data; the series did not exist yet. The gaps are not interpolated, since there is no known value before them.

In [7]:
saknas = konj.loc[konj["Värde"].isna()]
hal = saknas.groupby("indikator").agg(
    antal_manader=("månad", "size"),
    forsta=("månad", "min"),
    sista=("månad", "max"),
)
display(hal)

,antal_manader,forsta,sista
indikator,,,
Arbetade timmar,12,2000M01,2000M12
Näringslivets efterfrågan,23,2000M01,2001M11
Sysselsättning,12,2000M01,2000M12


## 8. From month to year

Fertility is measured per year, so the business cycle must also be annual. The annual value is **the mean of the year's 12 months**.

A year is only counted if **all 12 months are present**. Otherwise the annual value becomes `NaN`. A mean of just a few months would look as reliable as a full year without being so. *Näringslivets efterfrågan* in 2001, for example, is based only on December and would describe the end of the year rather than the whole year.

In [8]:
arsvarden = (
    konj.groupby(["indikator", "ar"])["Värde"]
    .agg(medel="mean", antal_manader="count")
    .reset_index()
)
arsvarden.loc[arsvarden["antal_manader"] < 12, "medel"] = float("nan")

ofullstandiga = arsvarden.loc[arsvarden["antal_manader"] < 12, ["indikator", "ar", "antal_manader"]]
print("Incomplete years set to NaN:")
display(ofullstandiga)

Incomplete years set to NaN:


,indikator,ar,antal_manader
0,Arbetade timmar,2000,0
208,Näringslivets efterfrågan,2000,0
209,Näringslivets efterfrågan,2001,1
260,Sysselsättning,2000,0


## 9. Wide format and short column names

The table is reshaped to **one row per year** with one column per indicator. The indicator names are translated to short ASCII names via an explicit mapping. The mapping also documents which name belongs to which indicator. The check makes sure that no indicator is lost or lacks a translation.

In [9]:
KOLUMNNAMN = {
    "Arbetade timmar": "arbetade_timmar",
    "BNP-indikator månad": "bnp_indikator",
    "Detaljhandel sällanköpsvaror": "detaljhandel_sallankop",
    "Hushållens konfidensindikator": "hushallens_konfidens",
    "Hushållens konsumtion": "hushallens_konsumtion",
    "Industrins orderingång": "industrins_orderingang",
    "Nyregistrerade lastbilar": "nyreg_lastbilar",
    "Nyregistrerade personbilar": "nyreg_personbilar",
    "Näringslivets efterfrågan": "naringslivets_efterfragan",
    "Näringslivets produktion": "naringslivets_produktion",
    "Sysselsättning": "sysselsattning",
    "Varuexport": "varuexport",
    "Varuimport": "varuimport",
}
assert set(KOLUMNNAMN) == set(arsvarden["indikator"])

konj_ar = (
    arsvarden.pivot(index="ar", columns="indikator", values="medel")
    .rename(columns=KOLUMNNAMN)
    .rename_axis(columns=None)
    .reset_index()
    [["ar", *KOLUMNNAMN.values()]]
)
display(konj_ar.round(2).head())
print("Shape:", konj_ar.shape)

,ar,arbetade_timmar,bnp_indikator,detaljhandel_sallankop,hushallens_konfidens,hushallens_konsumtion,industrins_orderingang,nyreg_lastbilar,nyreg_personbilar,naringslivets_efterfragan,naringslivets_produktion,sysselsattning,varuexport,varuimport
0,2000,NaN,0.08,0.52,1.79,0.45,0.51,0.52,0.95,NaN,0.20,NaN,0.45,0.98
1,2001,1.39,-0.14,-0.10,0.35,0.09,-0.62,-0.16,-0.76,NaN,0.03,0.80,-0.51,-0.16
2,2002,0.25,-0.42,-0.36,0.36,-0.24,-0.99,-0.50,-0.44,0.06,-0.47,0.51,-0.95,-0.98
3,2003,-1.03,-0.92,-0.61,0.01,-0.55,-0.84,-0.66,-0.00,-1.15,-0.63,-0.12,-0.79,-0.74
4,2004,-0.35,-0.26,-0.43,0.38,-0.93,-0.09,-0.31,0.13,-0.69,-0.30,-1.00,0.11,-0.43


Shape: (26, 14)


## 10. Final check

We check that each year 2000–2025 appears exactly once and that there are only 4 missing values. These are the years before the series started (see steps 7–8).

**To keep in mind in the analysis:** the business cycle is **national**. All municipalities get the same value in a given year, so there are only 26 independent observations of the business cycle, not one per municipality and year. That is why no lag is applied here. How many years the business cycle should precede fertility is a modelling choice that belongs to the analysis. It is done when the tables are joined, for example `konj_ar.assign(ar=konj_ar["ar"] + 1)` for a one-year lag.

In [10]:
assert konj_ar["ar"].tolist() == list(range(FORSTA_AR, SISTA_AR + 1))

saknade = konj_ar.set_index("ar").isna().stack()
saknade = saknade[saknade].index.tolist()
print("Missing annual values:", saknade)
assert len(saknade) == 4

Missing annual values: [(2000, 'arbetade_timmar'), (2000, 'naringslivets_efterfragan'), (2000, 'sysselsattning'), (2001, 'naringslivets_efterfragan')]


## 11. Save

In [11]:
utfil = PROCESSED / "konjunktur_ar.csv"
konj_ar.to_csv(utfil, index=False, encoding="utf-8-sig")

print(f"Saved {len(konj_ar)} rows × {konj_ar.shape[1] - 1} indicators to {utfil.relative_to(BAS)}")

Saved 26 rows × 13 indicators to data/processed/konjunktur_ar.csv
